# Place/transition nets

The classical net: places hold tokens, transitions move them along weighted arcs. We analyse the *water* net (`2 H2 + O2 -> 2 H2O`): reachability, boundedness, liveness and place invariants.

```mermaid
flowchart LR
    H2((H2 4)) -- 2 --> react[react]
    O2((O2 2)) --> react
    react -- 2 --> H2O((H2O 0))
```

Place invariants are integer vectors `y` with `y . C = 0`: the weighted token sum is constant in every reachable marking (conservation of atoms here).

In [1]:
from petrilab.petri import analysis, examples
from petrilab.petri.invariants import is_place_invariant, place_invariants, weighted_sum
from petrilab.petri.snakes_bridge import explore, to_snakes

n = examples.water()
graph = n.reachable()
for m, succ in graph.items():
    print(n.marking_dict(m), "->", [(t, n.marking_dict(x)) for t, x in succ])

{'H2': 4, 'O2': 2, 'H2O': 0} -> [('react', {'H2': 2, 'O2': 1, 'H2O': 2})]
{'H2': 2, 'O2': 1, 'H2O': 2} -> [('react', {'H2': 0, 'O2': 0, 'H2O': 4})]
{'H2': 0, 'O2': 0, 'H2O': 4} -> []


The reachability graph is a chain of three markings: `(4,2,0) -> (2,1,2) -> (0,0,4)`. Only `react` ever fires.

In [2]:
dead = [n.marking_dict(m) for m, s in graph.items() if not s]
print("dead markings:", dead)
print("target reachable:", n.marking(H2O=4) in graph)
print("(1,1,2) reachable:", n.marking(H2=1, O2=1, H2O=2) in graph)

dead markings: [{'H2': 0, 'O2': 0, 'H2O': 4}]
target reachable: True
(1,1,2) reachable: False


The only dead marking is all water (`H2O=4`), which is the intended end state. A marking that would need a half-fired reaction is not reachable.

In [3]:
space = explore(to_snakes(n))
print("bounded:", analysis.is_bounded(space))
print("live:", analysis.is_live(space, set(n.transitions)))
print("deadlock free:", analysis.is_deadlock_free(space))

bounded: True
live: False
deadlock free: False


The net is bounded (finite state space) but not live: once the hydrogen is used up, `react` can never fire again, so the net ends in a deadlock. For a terminating process that is expected.

In [4]:
print(is_place_invariant(n, {"H2": 1, "H2O": 1}))
print(is_place_invariant(n, {"O2": 2, "H2O": 1}))
print(is_place_invariant(n, {"H2": 1, "O2": 1}))

True
True
False


`H2 + H2O` (hydrogen atoms / 2) and `2*O2 + H2O` (oxygen atoms) are conserved; `H2 + O2` is not.

In [5]:
invs = place_invariants(n)
print(n.place_names, invs)
for y in invs:
    w = dict(zip(n.place_names, y, strict=True))
    print(w, {weighted_sum(n, w, m) for m in graph})

['H2', 'O2', 'H2O'] [[-1, 2, 0], [1, 0, 1]]
{'H2': -1, 'O2': 2, 'H2O': 0} {0}
{'H2': 1, 'O2': 0, 'H2O': 1} {4}


`place_invariants` computes an integer basis from the nullspace of the incidence matrix (two vectors here). Evaluating each weighted sum over all reachable markings gives a single value, confirming that the invariants are constant.